# Volume 9, Chapter 8: Semisimple Algebras and Induction (Python companion)

Wedderburn's theorem for S4 as a non-commutative Fourier transform, induced characters and Frobenius reciprocity, and Diaconis' spectral analysis of ranked data.

*Before each solution, predict what the output will look like.*

## Exercises

**P8.1 (★★) Wedderburn's theorem for $\mathfrak S_4$, by computer.** Construct the five irreducible representations of $\mathfrak S_4$ by explicit orthogonal matrices: trivial, sign, the $3$-dimensional standard representation (permutation matrices on the vectors with coordinate sum $0$), its product with the sign, and the $2$-dimensional one through $\mathfrak S_4\to\mathfrak S_3$ (the action on the three ways of splitting $\{1,2,3,4\}$ into two pairs). (a) Check that they are homomorphisms, irreducible and pairwise non-isomorphic, with characters. (b) Check that the map $\C[\mathfrak S_4]\to\prod M_{d_i}(\C)$ of Corollary 8.4 is bijective and turns convolution into blockwise products, and that the central idempotents $e_i$ act as claimed.

**P8.2 (★) Induction and Frobenius reciprocity.** For the subgroups $\mathfrak S_3$, $\langle(1234)\rangle$, $V_4$ and $\mathfrak S_2\times\mathfrak S_2$ of $\mathfrak S_4$, compute the induced characters of all irreducible characters of the subgroup with Proposition 8.8, decompose them, and check Frobenius reciprocity (Theorem 8.7). Count double cosets and compare with Exercise 8.1.

**P8.3 (★★) CS application: spectral analysis of rankings.** A survey in which $N$ people rank $n$ items gives a function $f$ on $\mathfrak S_n$ ($f(\sigma)$ = number of people giving ranking $\sigma$). Persi Diaconis proposed in the 1980s to analyse such data by its Fourier transform $\hat f(\rho)=\sum_\sigma f(\sigma)\rho(\sigma)$, the image of $f\in\C[\mathfrak S_n]$ under Wedderburn's isomorphism: the trivial component is the sample size, the standard representation holds the "first-order" information (how often item $i$ is ranked in position $j$), and higher components hold interactions. (a) Simulate rankings of $4$ items from a Mallows model $P(\sigma)\propto\theta^{\operatorname{inv}(\sigma)}$, and from a mixture of two Mallows models with opposite centres. (b) Compute the share of $\sum_\sigma|f(\sigma)-\bar f|^2$ carried by each irreducible representation (Plancherel), and interpret.

## Solutions

### P8.1

In [ ]:
import numpy as np, itertools
from numpy.linalg import matrix_rank, qr

n = 4
G = list(itertools.permutations(range(n)))
idx = {g: i for i, g in enumerate(G)}
def mul(a, b): return tuple(a[b[i]] for i in range(n))        # (ab)(i) = a(b(i))
def inv(a):
    r = [0] * n
    for i, x in enumerate(a): r[x] = i
    return tuple(r)
def sign(a):
    s = 1
    for i in range(n):
        for j in range(i + 1, n):
            if a[i] > a[j]: s = -s
    return s
def permmat(a, m):
    P = np.zeros((m, m))
    for i in range(m): P[a[i], i] = 1
    return P

def sumzero_basis(m):
    A = np.eye(m)[:, :m - 1] - np.eye(m)[:, 1:m]           # e_i - e_{i+1}
    return qr(A)[0]                                          # orthonormal basis of the sum-zero hyperplane

B4, B3 = sumzero_basis(4), sumzero_basis(3)
pairings = [((0, 1), (2, 3)), ((0, 2), (1, 3)), ((0, 3), (1, 2))]
def on_pairings(a):
    img = []
    for P in pairings:
        Q = tuple(sorted(tuple(sorted((a[x], a[y]))) for x, y in P))
        img.append(pairings.index(Q))
    return tuple(img)

reps = {
    "trivial":  lambda a: np.eye(1),
    "sign":     lambda a: np.eye(1) * sign(a),
    "chi2 (dim 2)": lambda a: B3.T @ permmat(on_pairings(a), 3) @ B3,
    "chi3 (standard)": lambda a: B4.T @ permmat(a, 4) @ B4,
    "chi3' = sign x standard": lambda a: sign(a) * (B4.T @ permmat(a, 4) @ B4),
}
for name, r in reps.items():
    hom = all(np.allclose(r(mul(a, b)), r(a) @ r(b)) for a in G for b in G)
    chi = np.array([np.trace(r(a)) for a in G])
    print(f"{name:26s} homomorphism: {hom},  <chi,chi> = {chi @ chi / 24:.3f}")
chars = {k: np.array([np.trace(r(a)) for a in G]) for k, r in reps.items()}
gram = np.array([[chars[a] @ chars[b] / 24 for b in chars] for a in chars])
print("Gram matrix of the characters:\n", np.round(gram, 6))

In [ ]:
def fourier(f):
    """f: vector indexed by G -> list of matrices f^(rho) = sum f(g) rho(g)."""
    return [sum(f[idx[g]] * r(g) for g in G) for r in reps.values()]

# the map C[S4] -> prod M_d as a 24 x 24 matrix: column for each basis element g
cols = [np.concatenate([M.ravel() for M in fourier(np.eye(24)[idx[g]])]) for g in G]
W = np.array(cols).T
print("dimension of the target:", W.shape[0], "  rank of the map:", matrix_rank(W))

def convolve(f, h):
    out = np.zeros(24)
    for g in G:
        for x in G:
            out[idx[mul(g, x)]] += f[idx[g]] * h[idx[x]]
    return out
rng = np.random.default_rng(8)
f, h = rng.normal(size=24), rng.normal(size=24)
print("convolution becomes blockwise product:",
      all(np.allclose(A, B @ C) for A, B, C in zip(fourier(convolve(f, h)), fourier(f), fourier(h))))
for i, (name, r) in enumerate(reps.items()):
    d = r(G[0]).shape[0]
    e = np.array([d / 24 * np.trace(r(inv(g))) for g in G])
    images = fourier(e)
    print(f"e_{name:24s} acts as", [("I" if np.allclose(M, np.eye(M.shape[0])) else "0" if np.allclose(M, 0) else "?") for M in images])

The five matrices are homomorphisms with orthonormal characters, so they are the five irreducible representations; the map to $\C\times\C\times M_2(\C)\times M_3(\C)\times M_3(\C)$ has rank $24=1+1+4+9+9$, turns convolution into blockwise matrix multiplication, and each $e_i$ is the identity in its own block and $0$ elsewhere, as Corollary 8.4 says. This is the *non-commutative Fourier transform*: for an abelian group all blocks are $1\times1$ and it is the discrete Fourier transform of Volume 2.

### P8.2

In [ ]:
from fractions import Fraction
def classes_of(H):
    seen, cls = set(), []
    for h in H:
        if h in seen: continue
        c = {mul(mul(x, h), inv(x)) for x in H}; seen |= c; cls.append(c)
    return cls

def inner(a, b, H): return (a @ np.conj(b)).real / len(H)

def induced_char(psi, H):
    Hset = set(H); hidx = {h: i for i, h in enumerate(H)}
    out = np.zeros(24, dtype=complex)
    for g in G:
        s = 0
        for x in G:
            y = mul(mul(x, g), inv(x))
            if y in Hset: s += psi[hidx[y]]
        out[idx[g]] = s / len(H)
    return out

def generated(gens):
    S = {tuple(range(n))}; frontier = list(S)
    while frontier:
        new = []
        for a in frontier:
            for g in gens:
                b = mul(a, g)
                if b not in S: S.add(b); new.append(b)
        frontier = new
    return sorted(S)

subgroups = {
    "S3 (fixing 4)": generated([(1, 0, 2, 3), (1, 2, 0, 3)]),
    "C4 = <(1234)>": generated([(1, 2, 3, 0)]),
    "V4":            generated([(1, 0, 3, 2), (2, 3, 0, 1)]),
    "S2 x S2":       generated([(1, 0, 2, 3), (0, 1, 3, 2)]),
}

def power_index(H, gens):
    """Write each h in H as a word gens[0]^x gens[1]^y ... (H abelian, generated by gens): exponent vectors."""
    out = {tuple(range(n)): (0,) * len(gens)}
    frontier = [tuple(range(n))]
    while frontier:
        new = []
        for h in frontier:
            for i, g in enumerate(gens):
                k = mul(h, g)
                if k not in out:
                    e = list(out[h]); e[i] += 1; out[k] = tuple(e); new.append(k)
        frontier = new
    return out

def order(g):
    k, x = 1, g
    while x != tuple(range(n)): x = mul(x, g); k += 1
    return k

def abelian_chars(H, gens):
    """All characters of an abelian group H generated by gens (orders of gens multiply to |H|)."""
    ex = power_index(H, gens); ords = [order(g) for g in gens]
    out = []
    for ks in itertools.product(*[range(o) for o in ords]):
        out.append(np.array([np.prod([np.exp(2j * np.pi * k * e / o) for k, e, o in zip(ks, ex[h], ords)]) for h in H]))
    return out

def irr_chars(name, H):
    if name.startswith("S3"):
        res = {k: np.array([chars[k][idx[h]] for h in H]) for k in chars}
        triv, sgn = res["trivial"], res["sign"]
        return [triv, sgn, res["chi3 (standard)"] - triv]        # Res chi3 = 1 + standard of S3
    gens = {"C4": [(1, 2, 3, 0)], "V4": [(1, 0, 3, 2), (2, 3, 0, 1)], "S2": [(1, 0, 2, 3), (0, 1, 3, 2)]}[name.split()[0]]
    return abelian_chars(H, gens)

for name, H in subgroups.items():
    irr = irr_chars(name, H)
    ok_recip = True
    lines = []
    for psi in irr:
        ind = induced_char(psi, H)
        mult = [round(inner(ind, chars[k], G), 6) for k in chars]
        for k in chars:
            res = np.array([chars[k][idx[h]] for h in H])
            ok_recip &= abs(inner(ind, chars[k], G) - inner(res, psi, H)) < 1e-9
        lines.append(f"     psi of degree {psi[H.index(tuple(range(n)))].real:.0f}: Ind = " +
                     " + ".join(f"{int(round(m))}*{k.split()[0]}" for m, k in zip(mult, chars) if round(m)))
    Hs = set(H)
    dc = len({frozenset(mul(mul(h, s), k) for h in H for k in H) for s in G})
    ind1 = induced_char(np.ones(len(H)), H)
    print(f"{name}: order {len(H)}, {len(irr)} irreducible characters, reciprocity holds: {ok_recip}")
    print("\n".join(lines))
    print(f"     double cosets H\\S4/H: {dc},  <Ind 1, Ind 1> = {inner(ind1, ind1, G):.0f}")

Frobenius reciprocity holds for every pair, and in each case $\langle\operatorname{Ind}1,\operatorname{Ind}1\rangle$ equals the number of double cosets (Exercise 8.1). For $\mathfrak S_3$, the permutation representation on $4$ points is $1+\chi_3$, so $\mathfrak S_4$ is doubly transitive on $\{1,2,3,4\}$; inducing the sign gives $\varepsilon+\chi_3'$, and the $2$-dimensional character induces to $\chi_2+\chi_3+\chi_3'$. For the normal subgroup $V_4$, $\operatorname{Ind}1$ is the regular representation of $\mathfrak S_4/V_4\cong\mathfrak S_3$, that is $1+\varepsilon+2\chi_2$, and the three non-trivial characters of $V_4$, which are conjugate under $\mathfrak S_4$, all induce to the same representation $\chi_3+\chi_3'$, as Mackey's formula predicts. The cyclic group $C_4$ and the non-normal Klein group $\mathfrak S_2\times\mathfrak S_2$ have the same number of double cosets, $3$, but different induced representations: $1+\chi_2+\chi_3'$ versus $1+\chi_2+\chi_3$.

### P8.3

In [ ]:
def inversions(a): return sum(1 for i in range(n) for j in range(i + 1, n) if a[i] > a[j])

def mallows_sample(theta, centre, N, rng):
    w = np.array([theta ** inversions(mul(inv(centre), g)) for g in G]); w /= w.sum()
    return rng.multinomial(N, w).astype(float)

def energy_shares(f):
    """Plancherel: sum_g |f(g) - mean|^2 = (1/|G|) sum_rho d_rho ||f^(rho)||_F^2 over non-trivial rho."""
    fh = fourier(f)
    dims = [M.shape[0] for M in fh]
    e = np.array([d * np.sum(np.abs(M) ** 2) / 24 for d, M in zip(dims, fh)])
    centred = np.sum((f - f.mean()) ** 2)
    assert abs(e[1:].sum() - centred) < 1e-6 * max(1, centred)
    return {k.split()[0]: float(v) for k, v in zip(list(reps)[1:], np.round(e[1:] / e[1:].sum(), 3))}

rng = np.random.default_rng(1)
identity, reverse = (0, 1, 2, 3), (3, 2, 1, 0)
print("uniform rankings:        ", energy_shares(rng.multinomial(2000, np.ones(24) / 24).astype(float)))
print("Mallows, theta = 0.3:    ", energy_shares(mallows_sample(0.3, identity, 2000, rng)))
f_mix = mallows_sample(0.3, identity, 1000, rng) + mallows_sample(0.3, reverse, 1000, rng)
print("mixture of two opposite: ", energy_shares(f_mix))

(The trivial component, which is just $N^2/24$, is left out.) For uniform rankings the data are pure noise, and the energy is spread roughly in proportion to $d_\rho^2$, that is $1/23,4/23,9/23,9/23$ for $\varepsilon,\chi_2,\chi_3,\chi_3'$, up to sampling fluctuations (the sign component, a single number, fluctuates most). A single Mallows model puts most of its energy, about $64\%$, in the standard representation $\chi_3$: the data are well explained by first-order effects, "item $i$ tends to be ranked in position $j$". In the mixture of two opposite Mallows models the share of $\chi_3$ drops to about $41\%$ and $\chi_2$ becomes the largest component. Since the permutation representation on unordered pairs of positions is $1+\chi_3+\chi_2$, the $\chi_2$ part is the information about *which pair of items fills the top two places* that first-order summaries cannot see: here $\{1,2\}$ or $\{3,4\}$, rarely a mixed pair. This is Diaconis' point: the Fourier decomposition of ranked data separates effects of different orders, as the Fourier series of a function on the circle separates frequencies, and the theory behind it is Wedderburn's theorem.